In [ ]:
import os
import sys
import json
import subprocess
import pandas as pd

# Set project root (adjust if needed)
PROJECT_ROOT = r"c:\Users\admin\Desktop\Python\capstone-Blue_Alpha_1"

sys.path.append(PROJECT_ROOT)

DATA_CSV = os.path.join(PROJECT_ROOT, "data", "raw", "monthly_mocha.csv")
OUTPUT_DIR = os.path.join(PROJECT_ROOT, "data", "output")
os.makedirs(OUTPUT_DIR, exist_ok=True)

print("Project root:", PROJECT_ROOT)
print("Data file exists:", os.path.exists(DATA_CSV))

In [ ]:
channels = ["meta", "google", "snapchat", "tiktok"]

channels_json = json.dumps(channels)

tmp_out = os.path.join(OUTPUT_DIR, "test_run.csv")

cmd = [
    sys.executable,
    "-m",
    "src.run_meridian_once",
    "--csv", DATA_CSV,
    "--channels_json", channels_json,
    "--target_channel", "tiktok",
    "--mu", "-3.5",
    "--sigma", "0.5",
    "--dist", "LogNormal",
    "--out_csv", tmp_out,
    "--n_chains", "4",
    "--n_adapt", "1000",
    "--n_burnin", "1000",
    "--n_keep", "1000",
    "--seed", "0",
]

proc = subprocess.run(
    cmd,
    cwd=PROJECT_ROOT,
    capture_output=True,
    text=True,
)

print("RETURN CODE:", proc.returncode)

print("\n====== STDOUT ======\n")
print(proc.stdout)

print("\n====== STDERR ======\n")
print(proc.stderr)

In [ ]:
df = pd.read_csv(tmp_out)

print("Columns:")
print(df.columns.tolist())

print("\nQC Overall Status:")
print(df["qc_overall_status"].iloc[0])

print("\nQC Pass/Fail:")
print(df["qc_pass_fail"].iloc[0])

print("\nQC Text:")
print(df["qc_text"].iloc[0])

In [ ]:
spend_cols = [c + "_spend" for c in channels]

variation_df = pd.DataFrame({
    "mean_spend": df[spend_cols].mean(),
    "std_spend": df[spend_cols].std(),
})

variation_df["cv"] = variation_df["std_spend"] / variation_df["mean_spend"]

variation_df.sort_values("cv")